In [3]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
HRV_DIR = 'hrv_data' # Assumed directory name
PATIENT_INFO_PATH = 'patient_info.csv'
# !!! Check HRV file naming convention !!! (Example below)
HRV_FILENAME_TEMPLATE = "patient_hr_{}.csv"
# !!! VERIFY THIS by checking an HRV file !!!
HRV_METADATA_ROWS = 1 # Assuming 1 for now, adjust if necessary

# --- Load Patient Info ---
try:
    patient_df = pd.read_csv(PATIENT_INFO_PATH, sep=';')
    print(f"Loaded patient info: {patient_df.shape[0]} rows")
except FileNotFoundError:
    print(f"Error: Patient info file not found at {PATIENT_INFO_PATH}")
    exit()

# --- Filter Participants for HRV Model ---
hrv_participants = patient_df[
    (patient_df['HRV'] == 1) & # Filter for those with HRV data
    (patient_df['MED'] == 0)   # Filter for non-medicated
].copy()

num_adhd_hrv = hrv_participants[hrv_participants['ADHD'] == 1].shape[0]
num_control_hrv = hrv_participants[hrv_participants['ADHD'] == 0].shape[0]

print(f"\nFiltered down to {hrv_participants.shape[0]} non-medicated participants with HRV data.")
print(f"  - ADHD: {num_adhd_hrv}")
print(f"  - Controls: {num_control_hrv}")

hrv_participant_ids = hrv_participants['ID'].tolist()

# --- Process data for all HRV participants ---
all_hrv_features = []

for participant_id in hrv_participant_ids:
    print(f"\nProcessing HRV for Participant ID: {participant_id}")
    participant_features = {'ID': participant_id}
    has_hrv_data_processed = False

    # --- Load HRV (IBI) Data ---
    hrv_filename = HRV_FILENAME_TEMPLATE.format(participant_id)
    hrv_filepath = os.path.join(HRV_DIR, hrv_filename)
    try:
        df_hrv = pd.read_csv(hrv_filepath, sep=';',
                             skiprows=HRV_METADATA_ROWS,
                             header=None, # No header after skipping
                             names=['TIMESTAMP', 'IBI'], # Assign correct names
                             index_col=0, # Use Timestamp as index
                             parse_dates=[0], # Parse index as dates
                             dayfirst=False) # YYYY-MM-DD format is standard
        print(f"  Loaded HRV (IBI) data: {df_hrv.shape[0]} rows")

        # --- Basic HRV Cleaning ---
        if not df_hrv.empty:
            # Example: Remove potential initial artifact if value is very high
            if df_hrv['IBI'].iloc[0] > 2000: # Adjust threshold if needed
                 df_hrv = df_hrv.iloc[1:]

            # Remove physiologically implausible values
            original_count = len(df_hrv)
            df_hrv = df_hrv[(df_hrv['IBI'] > 300) & (df_hrv['IBI'] < 2000)] # Typical range in ms
            removed_count = original_count - len(df_hrv)
            if removed_count > 0:
                print(f"  Removed {removed_count} implausible IBI values.")

            # --- Basic HRV Feature Engineering ---
            # Need at least 2 points to calculate differences
            if len(df_hrv) > 1:
                # Calculate differences between successive *valid* IBIs
                ibi_diffs = df_hrv['IBI'].diff()
                # Calculate RMSSD (Root Mean Square of Successive Differences)
                rmssd = ((ibi_diffs**2).mean())**0.5 # Use .mean() to handle potential NaNs from diff()
                participant_features['hrv_rmssd'] = rmssd
                participant_features['hrv_mean_ibi'] = df_hrv['IBI'].mean()
                # SDNN (Standard Deviation of NN intervals) is std dev of IBI here
                participant_features['hrv_sdnn'] = df_hrv['IBI'].std()
                # *** NOTE: For more advanced/robust features (pNN50, frequency), use pyhrv/NeuroKit2 ***
                has_hrv_data_processed = True
            else:
                print("  Warning: Not enough valid IBI data after cleaning to calculate features.")
                participant_features['hrv_rmssd'] = np.nan
                participant_features['hrv_mean_ibi'] = np.nan
                participant_features['hrv_sdnn'] = np.nan
        else:
            print("  Warning: Empty HRV dataframe after loading/initial cleaning.")
            participant_features['hrv_rmssd'] = np.nan
            participant_features['hrv_mean_ibi'] = np.nan
            participant_features['hrv_sdnn'] = np.nan


    except FileNotFoundError:
        print(f"  Warning: HRV file not found: {hrv_filepath}")
    except Exception as e:
        print(f"  Error processing HRV data for {participant_id}: {e}")

     # --- Add static features and label (only if HRV data was processed) ---
    if has_hrv_data_processed:
        # Retrieve info for this participant from the HRV-specific filtered list
        participant_info = hrv_participants.loc[hrv_participants['ID'] == participant_id].iloc[0]
        participant_features['SEX'] = participant_info['SEX']
        participant_features['AGE_GROUP'] = participant_info['AGE']
        participant_features['ADHD_LABEL'] = participant_info['ADHD']
        all_hrv_features.append(participant_features)
    else:
        print(f"  Skipping participant {participant_id} for HRV model due to missing data or processing errors.")


# --- Create Final HRV Feature DataFrame ---
if all_hrv_features:
    hrv_feature_df = pd.DataFrame(all_hrv_features)
    hrv_feature_df.set_index('ID', inplace=True)

    print("\n--- HRV Feature Extraction Complete ---")
    print(f"Created HRV feature dataframe with shape: {hrv_feature_df.shape}")
    print(hrv_feature_df.head())
    print("\nMissing values per feature:")
    print(hrv_feature_df.isnull().sum())
else:
    print("\n--- No participants processed successfully for HRV model. Check paths, names, and data. ---")

Loaded patient info: 134 rows

Filtered down to 53 non-medicated participants with HRV data.
  - ADHD: 27
  - Controls: 26

Processing HRV for Participant ID: 3
  Loaded HRV (IBI) data: 99012 rows
  Removed 231 implausible IBI values.

Processing HRV for Participant ID: 5
  Loaded HRV (IBI) data: 77018 rows
  Removed 12288 implausible IBI values.

Processing HRV for Participant ID: 7
  Loaded HRV (IBI) data: 92721 rows
  Removed 848 implausible IBI values.

Processing HRV for Participant ID: 9
  Loaded HRV (IBI) data: 62110 rows
  Removed 13482 implausible IBI values.

Processing HRV for Participant ID: 11
  Loaded HRV (IBI) data: 110690 rows
  Removed 173 implausible IBI values.

Processing HRV for Participant ID: 12
  Loaded HRV (IBI) data: 85625 rows
  Removed 16306 implausible IBI values.

Processing HRV for Participant ID: 15
  Loaded HRV (IBI) data: 108869 rows
  Removed 51 implausible IBI values.

Processing HRV for Participant ID: 16
  Loaded HRV (IBI) data: 111397 rows
  Remov

In [4]:
hrv_feature_df.head()

,hrv_rmssd,hrv_mean_ibi,hrv_sdnn,SEX,AGE_GROUP,ADHD_LABEL
ID,,,,,,
3,113.900127,757.753143,149.305502,1,2,1
5,178.412877,815.005301,229.321058,1,1,1
7,119.886589,825.759845,166.462590,0,3,0
9,264.183119,777.948686,336.778091,1,1,0
11,39.804294,779.952404,154.872555,1,3,1
